# E0 — CNN 1D por época (Línea base)

**Proyecto:** Efecto del contexto temporal en la clasificación de estadios del sueño  
**Grupo 25 — Universidad de los Andes, 2026**  
**Responsable:** Omar Díaz Hernández

---

Este notebook entrena el experimento E0: CNN 1D sobre épocas individuales, sin contexto temporal.  
Es la línea base contra la que se comparan E1 (concatenación) y E2 (GRU).

**Protocolo:** 3 corridas con semillas 42, 123 y 2026. Cohorte común (`eligible_only=True`) en val/test/ST.

## 0. Entorno y dependencias

In [ ]:
# Verificar GPU disponible
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Instalar dependencias faltantes en Colab
%pip install -q mlflow huggingface_hub scikit-learn

## 1. Clonar el repositorio y situarse en la rama correcta

In [ ]:
import os

REPO_URL    = "https://github.com/amquinteroc1/maia-grupo25-proyecto.git"
REPO_BRANCH = "training/e0-e1-e2"
REPO_DIR    = "/content/maia-grupo25-proyecto"

if not os.path.exists(REPO_DIR):
    !git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
!git log --oneline -3
print(f"\nDirectorio de trabajo: {os.getcwd()}")

In [ ]:
# Agregar la raíz del repo al path para importar src.*
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Verificar imports
from src.config import PATHS, CNN_ARCH, TRAINING, SEEDS, SEED_LABELS, N_CLASSES, CLASS_NAMES, MLFLOW
print("✓ src.config importado")
print(f"  Semillas: {SEEDS}")
print(f"  max_epochs: {TRAINING['max_epochs']}")
print(f"  early_stopping_patience: {TRAINING['early_stopping_patience']} (PROVISIONAL)")

## 2. Descargar datos desde HuggingFace

> Los datos ya están normalizados (Diego). **No volver a normalizar.**

In [ ]:
# Login a HuggingFace (repo privado)
from huggingface_hub import login
login()   # abre un prompt para pegar el token HF

In [ ]:
from huggingface_hub import snapshot_download

HF_REPO     = "maia-grupo25/sleep-edfx-data"
LOCAL_DATA  = REPO_DIR + "/data/processed"

os.makedirs(LOCAL_DATA, exist_ok=True)

snapshot_download(
    repo_id   = HF_REPO,
    repo_type = "dataset",
    local_dir = LOCAL_DATA,
)
print("\nArchivos descargados:")
!find {LOCAL_DATA} -name "*.npy" | sort | head -30

## 3. Verificar estructura de datos

In [ ]:
import numpy as np

data_e0  = PATHS["data_e0"]
data_seq = PATHS["data_seq"]

for split in ["train", "val", "test", "st"]:
    y = np.load(data_e0 / f"y_{split}.npy")
    g = np.load(data_e0 / f"groups_{split}.npy")
    print(f"E0  {split:5s}: {len(y):>8,} épocas | {len(np.unique(g))} sujetos")

print()
for split in ["train", "val", "test", "st"]:
    y = np.load(data_seq / f"y_seq_{split}.npy")
    e = np.load(data_seq / f"eligible_indices_{split}.npy")
    print(f"Seq {split:5s}: {len(y):>8,} épocas | eligible (cohorte común): {len(e):,}")

## 4. Configurar MLflow

In [ ]:
import mlflow

MLFLOW_DIR = str(PATHS["mlruns"])
mlflow.set_tracking_uri(f"file://{MLFLOW_DIR}")
mlflow.set_experiment(MLFLOW["experiment_name"])
print(f"MLflow tracking: {MLFLOW_DIR}")
print(f"Experimento   : {MLFLOW['experiment_name']}")

## 5. Funciones auxiliares

In [ ]:
import random
import torch

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    print(f"  Semilla fijada: {seed}")

def get_device() -> torch.device:
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
    else:
        dev = torch.device("cpu")
        print("  CPU (no GPU disponible)")
    return dev

## 6. Corrida exploratoria (E0, semilla 42)

Objetivo: observar curvas de aprendizaje para **decidir `max_epochs` y `early_stopping_patience` finales** antes de lanzar las 9 corridas completas.

> Ejecutar esta celda primero. Revisar en MLflow si la curva de `val_f1_macro` ya convergió antes de la época 100.

In [ ]:
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader

from src.models.cnn1d import SleepCNN1D
from src.training.dataset import SleepDatasetE0, compute_class_weights
from src.training.trainer import run_training

SEED_EXPLORATORIO = 42

# Para la corrida exploratoria desactivamos el early stopping
# poniendo una paciencia igual al máximo de épocas
TRAINING_EXPLORATORIO = dict(TRAINING)
TRAINING_EXPLORATORIO["early_stopping_patience"] = TRAINING["max_epochs"]

print("=" * 55)
print(f"CORRIDA EXPLORATORIA — E0 semilla {SEED_EXPLORATORIO}")
print(f"max_epochs : {TRAINING['max_epochs']}")
print(f"early stop : DESACTIVADO (paciencia = max_epochs)")
print("=" * 55)

set_seed(SEED_EXPLORATORIO)
device = get_device()

# Datasets
train_ds = SleepDatasetE0(split="train", eligible_only=False)
val_ds   = SleepDatasetE0(split="val",  eligible_only=True)
test_ds  = SleepDatasetE0(split="test", eligible_only=True)
st_ds    = SleepDatasetE0(split="st",   eligible_only=True)

print(f"\nTamaños:")
print(f"  SC-train : {len(train_ds):,}")
print(f"  SC-val   : {len(val_ds):,}  (cohorte común)")
print(f"  SC-test  : {len(test_ds):,}  (cohorte común)")
print(f"  ST       : {len(st_ds):,}  (cohorte común)")

pin = TRAINING["pin_memory"] and device.type == "cuda"
bs  = TRAINING["batch_size"]

train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,  num_workers=0, pin_memory=pin)
val_loader   = DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=0, pin_memory=pin)
test_loader  = DataLoader(test_ds,  batch_size=bs, shuffle=False, num_workers=0, pin_memory=pin)
st_loader    = DataLoader(st_ds,    batch_size=bs, shuffle=False, num_workers=0, pin_memory=pin)

# Pesos de clase
class_weights = compute_class_weights(train_ds.get_labels(), n_classes=N_CLASSES).to(device)
print(f"\nPesos: { {n: round(float(w), 3) for n, w in zip(CLASS_NAMES, class_weights.cpu())} }")

# Modelo
model = SleepCNN1D(
    n_channels=CNN_ARCH["n_channels"],
    n_classes=N_CLASSES,
    dropout=CNN_ARCH["dropout"],
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=TRAINING["lr"], weight_decay=TRAINING["weight_decay"])
scheduler = ReduceLROnPlateau(optimizer, mode="max",
                               patience=TRAINING["scheduler_patience"],
                               factor=TRAINING["scheduler_factor"],
                               min_lr=TRAINING["scheduler_min_lr"])
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Patch temporal de paciencia
original_patience = TRAINING["early_stopping_patience"]
TRAINING["early_stopping_patience"] = TRAINING["max_epochs"]

result_exploratorio = run_training(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    experiment="E0-exploratorio",
    seed=SEED_EXPLORATORIO,
    run_dir=PATHS["experiments"] / "E0" / "exploratorio",
    groups_test=test_ds.get_groups(),
    st_loader=st_loader,
    groups_st=st_ds.get_groups(),
)

TRAINING["early_stopping_patience"] = original_patience

print(f"\n✓ Exploratoria completada")
print(f"  Mejor época    : {result_exploratorio['best_epoch']}")
print(f"  val_f1_macro   : {result_exploratorio['best_val_f1']:.4f}")
print(f"  test_f1_macro  : {result_exploratorio['test_metrics']['test_f1_macro']:.4f}")
print(f"  test_f1_N1     : {result_exploratorio['test_metrics']['test_f1_n1']:.4f}")

### 6.1 Revisar curvas de aprendizaje

Antes de continuar, revisa en MLflow la curva de `val_f1_macro` y decide:
- ¿En qué época convergió? → ese valor + 10 es un buen candidato para `max_epochs`
- ¿Cuántas épocas pasaron sin mejora al final? → ese número + 2 es un buen candidato para `early_stopping_patience`

Actualiza `config.py` con los valores definitivos antes de correr la celda siguiente.

In [ ]:
# Visualizar curva de val_f1_macro desde el history guardado
import matplotlib.pyplot as plt

history = result_exploratorio["history"]
epochs  = [h["epoch"]     for h in history]
val_f1  = [h["val_f1"]    for h in history]
val_n1  = [h["val_f1_n1"] for h in history]
tr_loss = [h["tr_loss"]   for h in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(epochs, val_f1, label="val_f1_macro")
axes[0].plot(epochs, val_n1, label="val_f1_N1", linestyle="--")
axes[0].axvline(result_exploratorio["best_epoch"], color="red", linestyle=":", label=f"mejor época ({result_exploratorio['best_epoch']})")
axes[0].set_title("F1 Macro y F1 N1 en validación")
axes[0].set_xlabel("Época")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs, tr_loss, color="orange", label="train_loss")
axes[1].set_title("Train loss")
axes[1].set_xlabel("Época")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(str(PATHS["experiments"] / "E0" / "exploratorio" / "curvas_exploratorio.png"), dpi=120)
plt.show()
print(f"Mejor época: {result_exploratorio['best_epoch']} — val_f1_macro: {result_exploratorio['best_val_f1']:.4f}")

## 7. Corridas oficiales E0 — 3 semillas

> **Ejecutar solo después de revisar la corrida exploratoria y actualizar `config.py`.**

In [ ]:
# Reimportar config para tomar los valores definitivos que ajustaste
import importlib
import src.config as cfg_module
importlib.reload(cfg_module)
from src.config import TRAINING, SEEDS, SEED_LABELS

print("Configuración definitiva:")
print(f"  max_epochs              : {TRAINING['max_epochs']}")
print(f"  early_stopping_patience : {TRAINING['early_stopping_patience']}")
print(f"  scheduler_patience      : {TRAINING['scheduler_patience']}")

In [ ]:
all_results = {}

for seed in SEEDS:
    label = SEED_LABELS[seed]
    print(f"\n{'='*55}")
    print(f"  E0 — Semilla {seed} (Condición {label})")
    print(f"{'='*55}")

    set_seed(seed)
    device = get_device()

    train_ds = SleepDatasetE0(split="train", eligible_only=False)
    val_ds   = SleepDatasetE0(split="val",   eligible_only=True)
    test_ds  = SleepDatasetE0(split="test",  eligible_only=True)
    st_ds    = SleepDatasetE0(split="st",    eligible_only=True)

    pin = TRAINING["pin_memory"] and device.type == "cuda"
    bs  = TRAINING["batch_size"]

    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,  num_workers=0, pin_memory=pin,
                              worker_init_fn=lambda wid: np.random.seed(seed + wid))
    val_loader   = DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=0, pin_memory=pin)
    test_loader  = DataLoader(test_ds,  batch_size=bs, shuffle=False, num_workers=0, pin_memory=pin)
    st_loader    = DataLoader(st_ds,    batch_size=bs, shuffle=False, num_workers=0, pin_memory=pin)

    class_weights = compute_class_weights(train_ds.get_labels(), n_classes=N_CLASSES).to(device)

    model = SleepCNN1D(
        n_channels=CNN_ARCH["n_channels"],
        n_classes=N_CLASSES,
        dropout=CNN_ARCH["dropout"],
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=TRAINING["lr"], weight_decay=TRAINING["weight_decay"])
    scheduler = ReduceLROnPlateau(optimizer, mode="max",
                                   patience=TRAINING["scheduler_patience"],
                                   factor=TRAINING["scheduler_factor"],
                                   min_lr=TRAINING["scheduler_min_lr"])
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    result = run_training(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        experiment="E0",
        seed=seed,
        run_dir=PATHS["experiments"] / "E0" / f"seed{seed}",
        groups_test=test_ds.get_groups(),
        st_loader=st_loader,
        groups_st=st_ds.get_groups(),
    )
    all_results[seed] = result

## 8. Resumen final E0

In [ ]:
print("\n" + "="*55)
print("RESUMEN E0 — 3 corridas")
print("="*55)

f1s = [r["test_metrics"]["test_f1_macro"] for r in all_results.values()]
n1s = [r["test_metrics"]["test_f1_n1"]    for r in all_results.values()]
kas = [r["test_metrics"]["test_kappa"]     for r in all_results.values()]
bas = [r["test_metrics"]["test_balanced_accuracy"] for r in all_results.values()]

print(f"  test_f1_macro     : {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
print(f"  test_f1_N1        : {np.mean(n1s):.4f} ± {np.std(n1s):.4f}")
print(f"  test_kappa        : {np.mean(kas):.4f} ± {np.std(kas):.4f}")
print(f"  test_balanced_acc : {np.mean(bas):.4f} ± {np.std(bas):.4f}")
print()

for seed, r in all_results.items():
    lbl = SEED_LABELS[seed]
    m   = r["test_metrics"]
    print(
        f"  [{lbl}] seed={seed} | "
        f"f1_macro={m['test_f1_macro']:.4f} | "
        f"f1_N1={m['test_f1_n1']:.4f} | "
        f"kappa={m['test_kappa']:.4f} | "
        f"ep={r['best_epoch']}"
    )

print("\nArtefactos para Luigi guardados en:")
for seed in SEEDS:
    print(f"  experiments/E0/seed{seed}/")